In [ ]:
%matplotlib qt5
import hyperspy.api as hs
import pyxem as pxm
import numpy as np
import orix
import matplotlib.pyplot as plt

from pathlib import Path
from diffpy.structure import Atom, Lattice, Structure
from diffsims.generators.simulation_generator import SimulationGenerator
from orix.vector import Miller, Vector3d
from orix.plot import IPFColorKeyTSL
from matplotlib.lines import Line2D
from dataclasses import dataclass

import plotly.graph_objects as go

In [ ]:
PATH_SPED5 = Path("D:/datasets/250225_Ag_oxww/20250225_175525/SPED5_calibrated.zspy")
PATH_SPED6 = Path("D:/datasets/250225_Ag_oxww/20250225_180346/SPED6_calibrated.zspy")
PATH_SPED7 = Path("D:/datasets/250225_Ag_oxww/20250225_181109/SPED7_calibrated.zspy")
PATH_SPED8 = Path("D:/datasets/250225_Ag_oxww/20250225_182247/SPED8_calibrated.zspy")

path_list = [PATH_SPED6, PATH_SPED7, PATH_SPED8]

colors = ["blue", "red", "green", "orange"]
labels = ["SPED5", "SPED6", "SPED7", "SPED8"]

In [ ]:
# Define the crystal structure for silver (Ag)
a = 4.08
atoms = [Atom('Ag', [0, 0, 0]), Atom('Ag', [0.5, 0.5, 0]), Atom('Ag', [0.5, 0, 0.5]), Atom('Ag', [0, 0.5, 0.5])]
lattice = Lattice(a, a, a, 90, 90, 90)
phase = orix.crystal_map.Phase(name='Ag', space_group=225, structure=Structure(atoms, lattice))

angular_resolution = 0.5
grid = orix.sampling.get_sample_reduced_fundamental(angular_resolution, point_group=phase.point_group)
orientations = orix.quaternion.Orientation(grid, symmetry=phase.point_group)

simgen = SimulationGenerator(precession_angle=1., minimum_intensity=1e-5)
simulations = simgen.calculate_diffraction2d(phase=phase, rotation=grid, reciprocal_radius=1.5, with_direct_beam=False, max_excitation_error=0.01)

# Create IPF color keys for x, y, and z directions
key_x = IPFColorKeyTSL(phase.point_group, Vector3d.xvector())
key_y = IPFColorKeyTSL(phase.point_group, Vector3d.yvector())
key_z = IPFColorKeyTSL(phase.point_group, Vector3d.zvector())

In [ ]:
def function1(path, masking=False):
    SPED_data = hs.load(path, lazy=True)
    SPED_data.change_dtype("float32")

    ny, nx = SPED_data.axes_manager.signal_shape
    center = (ny/2 - 0.5, nx/2 - 0.5)
    SPED_data.calibration(center=center)
    if masking == True:
        SPED_template = SPED_data.template_match_disk(disk_r=3, subtract_min=False)
        Peaks_data = SPED_template.get_diffraction_vectors(min_distance=3, threshold_abs=0.6)
        SPED_mask = Peaks_data.to_mask(disk_r=3)
        return SPED_data * SPED_mask
    else:
        return SPED_data

def function2(data):
    SPED_azi = data.get_azimuthal_integral2d(npt=256)
    results = SPED_azi.get_orientation(simulations, n_best=grid.size, frac_keep=0.5)
    results.compute()
    
    single_phase = results.to_single_phase_orientations()[:,:,0]
    xmap = results.to_crystal_map()
    oris = xmap.orientations
    corrs = results.data[:,:,0,1].flatten()
    
    return results, xmap, oris, corrs, single_phase

@dataclass
class SPEDResult:
    results: object
    xmap: object
    oris: object
    corrs: object
    single_phase: object   

    def colormap(self):
        oris_z = key_z.orientation2color(self.oris)
        self.xmap.plot(oris_z, overlay=self.corrs, remove_padding=True)
        oris_x = key_x.orientation2color(self.oris)
        self.xmap.plot(oris_x, overlay=self.corrs, remove_padding=True)
        oris_y = key_y.orientation2color(self.oris)
        self.xmap.plot(oris_y, overlay=self.corrs, remove_padding=True)
    
    def ipf_correlation(self, ix, iy):
        correlations_i = self.results.inav[ix, iy].data[:, 1]
        template_indices_i = (self.results.inav[ix, iy].data[:, 0]).astype('int16')
        orientations_i = orientations[template_indices_i]
        
        fig = plt.figure()
        ax = fig.add_subplot(111, projection="ipf", symmetry=phase.point_group)
        ax.scatter(orientations_i, c=correlations_i, cmap='inferno', picker=True)
        ax.scatter(orientations_i[0], c="r", marker="x")
        
        fig_dp, ax_dp = plt.subplots()
        first = template_indices_i[0]
        
        sim_i = simulations.irot[first]
        pat = sim_i.get_diffraction_pattern(
            shape=(256, 256),        # pick your detector size
            sigma=2,                 # spot size in pixels
            calibration=0.01,        # real->pixel scale (tune to match your data)
            direct_beam_position=(128, 128),
            normalize=True,
        )
        ax_dp.imshow(pat, origin="lower")

    def ipf_matches(self):
        fig = self.single_phase.scatter(projection="ipf", return_figure=True, label=labels[0])
        plt.show()

In [ ]:
sped5 = SPEDResult(*function2(function1(PATH_SPED5)))
sped6 = SPEDResult(*function2(function1(PATH_SPED6)))
sped7 = SPEDResult(*function2(function1(PATH_SPED7)))
sped8 = SPEDResult(*function2(function1(PATH_SPED8)))

sped_list = np.array([sped5, sped6, sped7, sped8])

In [ ]:
def ipf_matches_tilt_series(data_list):
    fig = data_list[0].single_phase.scatter(projection="ipf", return_figure=True, label=labels[0])

    for data, color, name in zip(data_list[1:], colors[1:], labels[1:]):
        data.single_phase.scatter(
            projection="ipf",
            figure=fig,
            c=color,
            label=name,
        )
    ax = fig.axes[0]
    ax.legend()
    plt.show()

ipf_matches_tilt_series(sped_list)

# Misorientations

## Test

In [ ]:
def two_areas_compare(path1, path2):
    first_sped = hs.load(path1)
    second_sped = hs.load(path2)

    reference_roi = hs.roi.RectangularROI()
    target_roi = hs.roi.RectangularROI()

    first_sped.plot()
    second_sped.plot()

    ref_roi_1, ref_roi_2 = reference_roi.interactive(first_sped), reference_roi.interactive(second_sped)
    tag_roi_1, tag_roi_2 = target_roi.interactive(first_sped, color='C1'), target_roi.interactive(second_sped, color='C1')

    plt.show(block=True)

    ref_mean_1 = ref_roi_1.mean((0,1))
    ref_mean_2 = ref_roi_2.mean((0,1))

    def template(ref_data, target_data):
        ref_data.change_dtype("float32")
        target_data.change_dtype("float32")
    
        ny, nx = target_data.axes_manager.signal_shape
        center = (ny/2 - 0.5, nx/2 - 0.5)
        
        ref_data.calibration(center=center)
        target_data.calibration(center=center)
        
        target_azi = target_data.get_azimuthal_integral2d(npt=256)
        ref_azi = ref_data.get_azimuthal_integral2d(npt=256)
        
        target_results = target_azi.get_orientation(simulations, n_best=grid.size, frac_keep=0.5)
        ref_results = ref_azi.get_orientation(simulations, n_best=grid.size, frac_keep=0.5)
             
        target_single_phase = target_results.to_single_phase_orientations()[:,:,:10]
        ref_single_phase = ref_results.to_single_phase_orientations()[0]
        
        target_xmap = target_results.to_crystal_map()
        target_oris = target_xmap.orientations
        target_corrs = target_results.data[:,:,0,1].flatten()
        
        result = SPEDResult(
            results=target_results, 
            xmap=target_xmap, 
            oris=target_oris, 
            corrs=target_corrs, 
            single_phase=target_single_phase)
        
        return result, ref_single_phase

    res1, single1 = template(ref_mean_1, tag_roi_1)
    res2, single2 = template(ref_mean_2, tag_roi_2)

    diff1 = res1.single_phase - single1
    diff2 = res2.single_phase - single2

    comparison = np.zeros(diff1.shape)
    for i in range(diff1.shape[2]):
        comparison[:,:,i] = diff1[:,:,0].angle_with(diff2[:,:,i], degrees=True)

    minimum = np.argmin(comparison, axis=2)

    shift = np.take_along_axis(
        res2.single_phase,
        (minimum - 1)[..., np.newaxis],
        axis=2
    )
    shift = shift.transpose(1, 0, 2)
    shift = shift.squeeze().flatten()
    
    res2.colormap()
    oris_z2 = key_z.orientation2color(shift)
    res2.xmap.plot(oris_z2, overlay=res2.corrs, remove_padding=True)
    oris_x2 = key_x.orientation2color(shift)
    res2.xmap.plot(oris_x2, overlay=res2.corrs, remove_padding=True)
    oris_y2= key_y.orientation2color(shift)
    res2.xmap.plot(oris_y2, overlay=res2.corrs, remove_padding=True)

two_areas_compare(PATH_SPED5, PATH_SPED6)

## Improve function

In [ ]:
def improve_function(base, data, select_amount=10):
    base_difference = (base.single_phase - base.single_phase[0,0]).map_into_symmetry_reduced_zone()
    data_single_phase = data.results.to_single_phase_orientations()[:,:,:select_amount]
    difference = (data_single_phase - data_single_phase[0,0]).map_into_symmetry_reduced_zone()
    
    array = []
    for i in range(select_amount):
        comparison = (difference[:,:,i].inv() * base_difference).angle
        array.append(comparison)
    
    stack = np.stack(array, axis=0)
    idx = np.argmin(stack, axis=0)
    
    result = np.take_along_axis(
        data_single_phase,
        (idx - 1)[..., None],
        axis=2
    )
    
    a = result.transpose(1, 0, 2)
    a = np.squeeze(a).flatten()

    oris_z = key_z.orientation2color(a)
    data.xmap.plot(oris_z, overlay=data.corrs, remove_padding=True)
    oris_x = key_x.orientation2color(a)
    data.xmap.plot(oris_x, overlay=data.corrs, remove_padding=True)
    oris_y = key_y.orientation2color(a)
    data.xmap.plot(oris_y, overlay=data.corrs, remove_padding=True)